# Track eta-phi maps

Reads and plots a 2D tracking monitoring element (by default `TrackEtaPhi_ImpactPoint_GenTk`, the number of high-purity tracks -- pt > 1 GeV -- as a function of eta and phi at the track's impact point) from DIALS.

Supports plotting individual lumisections, and also comparing a single "target" lumisection side by side with a "reference" built by summing a range of lumisections (or a full run) -- e.g. to compare a lumisection suspected of an anomaly against the run's typical behaviour -- plus a third panel with the ratio of the two after normalizing each to a common scale.

Note: unlike the pixel cluster occupancy monitoring elements used elsewhere in this repo (which live in the `ZeroBias`/`PromptReco` dataset), this tracking monitoring element was only found to be populated in the `StreamExpress` dataset -- not every lumisection has Express-stream data recorded, so a given (run, ls) may legitimately come back empty.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from cmsdials import Dials
from cmsdials.auth.bearer import Credentials
from cmsdials.filters import LumisectionHistogram2DFilters

In [ ]:
# authenticate; on first use this will print a device-login URL to visit.
# credentials are cached in .cache/ next to this notebook, so subsequent
# runs (within the token's validity) won't need to re-authenticate.
creds = Credentials.from_creds_file(cache_dir='.cache')
dials = Dials(creds, workspace='tracker')

In [ ]:
# --- configuration ---

# the monitoring element to plot.
# a few other candidates, in case this one turns out not to be the right one
# (see the notebook that identified it for how these were found):
#   Tracking/TrackParameters/highPurityTracks/pt_1/GeneralProperties/TrackEta_ImpactPoint_GenTk   (1D, eta only)
#   Tracking/TrackParameters/highPurityTracks/pt_1/GeneralProperties/TrackPhi_ImpactPoint_GenTk   (1D, phi only)
#   Tracking/TrackParameters/generalTracks/GeneralProperties/TrackEtaHighpurity_ImpactPoint_GenTk (1D, no explicit pt cut)
ME = 'Tracking/TrackParameters/highPurityTracks/pt_1/GeneralProperties/TrackEtaPhi_ImpactPoint_GenTk'

# dataset to query; a regex, since the exact processing version (v1, v2, ...)
# can differ per era. This ME was only found populated in StreamExpress, not
# ZeroBias/PromptReco (unlike the pixel cluster occupancy MEs used elsewhere
# in this repo) -- adjust if a different stream turns out to be needed.
DATASET_REGEX = 'StreamExpress'

# list of (run, lumisection) pairs to plot individually; edit freely.
RUNS_LS = [
    (381516, 532),
    (381516, 533),
]

# target-vs-reference-vs-ratio comparisons to run by default; each entry is
# (target_run, target_ls, ref_run, ref_ls_min, ref_ls_max) -- ref_run=None
# defaults to target_run, and ref_ls_min=ref_ls_max=None means the reference
# is built from the full run's statistics. Edit/extend freely.
COMPARISONS = [
    (392250, 633, 392250, None, None),
    (396871, 743, 396871, None, None),
]

# output directory for saved plots (see save_trackmap()/save_compare_trackmaps() below).
PLOT_DIR = 'trackmap_plots'

In [ ]:
def get_trackmap(run, ls, me=ME, dataset_regex=DATASET_REGEX):
    '''Fetches a single 2D histogram for the given (run, ls).
    Returns (data, x_min, x_max, y_min, y_max, entries, dataset) or None if not found.'''
    filters = LumisectionHistogram2DFilters(
        me=me,
        dataset__regex=dataset_regex,
        run_number=run,
        ls_number=ls,
    )
    result = dials.h2d.list(filters)
    if not result.results:
        print(f'WARNING: no data found for run {run}, LS {ls} (me={me}, dataset regex={dataset_regex}).')
        return None
    if len(result.results) > 1:
        datasets = [r.dataset for r in result.results]
        print(f'WARNING: run {run}, LS {ls} matched {len(result.results)} datasets ({datasets}); using the first one.')
    r = result.results[0]
    data = np.array(r.data)
    return data, r.x_min, r.x_max, r.y_min, r.y_max, r.entries, r.dataset

In [ ]:
def get_trackmap_sum(run, ls_min=None, ls_max=None, me=ME, dataset_regex=DATASET_REGEX):
    '''Fetches and sums 2D histograms over lumisections [ls_min, ls_max] in
    a single run (both None = the full run). Returns (summed_data, x_min,
    x_max, y_min, y_max, total_entries, n_ls, dataset), or None if nothing
    was found. n_ls is how many lumisections actually contributed (StreamExpress
    does not record every lumisection, so this can be fewer than ls_max-ls_min+1).'''
    filters = LumisectionHistogram2DFilters(
        me=me,
        dataset__regex=dataset_regex,
        run_number=run,
    )
    result = dials.h2d.list_all(filters, enable_progress=False)
    results = result.results
    if ls_min is not None:
        results = [r for r in results if r.ls_number >= ls_min]
    if ls_max is not None:
        results = [r for r in results if r.ls_number <= ls_max]
    if not results:
        msg = f'WARNING: no data found for run {run}'
        if ls_min is not None or ls_max is not None: msg += f', LS {ls_min}-{ls_max}'
        msg += f' (me={me}, dataset regex={dataset_regex}).'
        print(msg)
        return None
    data = np.sum([np.array(r.data) for r in results], axis=0)
    total_entries = sum(r.entries for r in results)
    r0 = results[0]
    return data, r0.x_min, r0.x_max, r0.y_min, r0.y_max, total_entries, len(results), r0.dataset

In [ ]:
def normalize_top_half(data):
    '''Normalizes a 2D histogram to a common scale, using only the top 50%
    of bins by value to define that scale -- so a handful of anomalously
    low bins (the very effect we want to characterize) don't skew the
    normalization used to compare it against another histogram. Returns
    (normalized_data, scale).'''
    flat = data.flatten()
    median = np.median(flat)
    top_half = flat[flat >= median]
    scale = np.mean(top_half) if len(top_half) else 1.0
    return (data / scale if scale else data), scale

In [ ]:
def _plot_trackmap_data(data, x_min, x_max, y_min, y_max, ax=None, year=None, run=None, title=None,
                         cmap='viridis', vmin=None, vmax=None, cbar_label='Number of tracks'):
    '''Shared plotting/styling logic (CMS-public-plot style: "CMS Preliminary"
    top-left, "<year> (13.6 TeV)" top-right) for a single already-fetched
    (possibly summed, possibly normalized) 2D histogram. Returns (fig, ax, im).'''
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    else:
        fig = ax.figure

    im = ax.imshow(
        data, origin='lower', aspect='equal',
        extent=[x_min, x_max, y_min, y_max],
        cmap=cmap, vmin=vmin, vmax=vmax,
    )
    # pin the axes limits to exactly the image extent, so neither the
    # header text below nor anything else causes matplotlib to autoscale
    # the axes beyond the actual data -- this is also what keeps the
    # colorbar (tied to the axes height) matching the plot instead of
    # stretching to fill extra whitespace.
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    ax.set_xlabel('Track $\\eta$', fontsize=14)
    ax.set_ylabel('Track $\\phi$', fontsize=14)
    ax.tick_params(labelsize=12)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(cbar_label, fontsize=14)
    cbar.ax.tick_params(labelsize=12)

    if year is None and run is not None:
        year = 2024 if run < 390000 else 2025
    ax.text(0.0, 1.01, 'CMS', fontsize=17, fontweight='bold', ha='left', va='bottom', transform=ax.transAxes)
    ax.text(0.16, 1.01, 'Preliminary', fontsize=13, style='italic', ha='left', va='bottom', transform=ax.transAxes)
    ax.text(1.0, 1.01, f'{year} (13.6 TeV)' if year else '(13.6 TeV)', fontsize=13, ha='right', va='bottom', transform=ax.transAxes)
    # optional descriptive label (e.g. distinguishing target vs reference in
    # a side-by-side comparison), placed above the CMS row so it doesn't clash
    if title:
        ax.text(0.5, 1.10, title, fontsize=11, ha='center', va='bottom', transform=ax.transAxes)

    return fig, ax, im

In [ ]:
def plot_trackmap(run, ls, ax=None, year=None, title=None, **kwargs):
    '''Fetches and plots a single (run, ls) track map. See _plot_trackmap_data().
    Returns (fig, ax, im), or None if no data was found.'''
    res = get_trackmap(run, ls, **kwargs)
    if res is None:
        return None
    data, x_min, x_max, y_min, y_max, entries, dataset = res
    return _plot_trackmap_data(data, x_min, x_max, y_min, y_max, ax=ax, year=year, run=run, title=title)

In [ ]:
def plot_trackmap_sum(run, ls_min=None, ls_max=None, ax=None, year=None, title=None, **kwargs):
    '''Fetches, sums and plots a track map over lumisections [ls_min, ls_max]
    in a single run (both None = the full run). See get_trackmap_sum().
    Returns (fig, ax, im), or None if no data was found.'''
    res = get_trackmap_sum(run, ls_min=ls_min, ls_max=ls_max, **kwargs)
    if res is None:
        return None
    data, x_min, x_max, y_min, y_max, entries, n_ls, dataset = res
    return _plot_trackmap_data(data, x_min, x_max, y_min, y_max, ax=ax, year=year, run=run, title=title)

In [ ]:
def compare_trackmaps(target_run, target_ls, ref_run=None, ref_ls_min=None, ref_ls_max=None,
                       ratio_vmin=0, ratio_vmax=2, sig_vmin=-10, sig_vmax=10,
                       me=ME, dataset_regex=DATASET_REGEX):
    '''Plots a single target lumisection, a reference built by summing
    lumisections [ref_ls_min, ref_ls_max] of ref_run (defaults to target_run;
    both None = that run's full statistics), a third panel with the ratio of
    the two after each has been normalized to a common scale (see
    normalize_top_half()) -- so the ratio is meaningful despite the target
    (1 LS) and reference (many summed LS) being at very different absolute
    scales -- and a fourth panel with the significance of the deviation,
    (norm_target - norm_ref) / sigma, where sigma is the Poisson uncertainty
    (sqrt of bin content) on each map, scaled by the same normalization
    factor as the nominal values and combined in quadrature. Returns
    (fig, axs), where axs has 4 (possibly-None-content) axes.'''
    if ref_run is None:
        ref_run = target_run

    target_res = get_trackmap(target_run, target_ls, me=me, dataset_regex=dataset_regex)
    ref_res = get_trackmap_sum(ref_run, ls_min=ref_ls_min, ls_max=ref_ls_max, me=me, dataset_regex=dataset_regex)

    fig, axs = plt.subplots(1, 4, figsize=(25, 6))

    if target_res is not None:
        data, x_min, x_max, y_min, y_max, entries, dataset = target_res
        _plot_trackmap_data(data, x_min, x_max, y_min, y_max, ax=axs[0], run=target_run,
                             title=f'Target: Run {target_run}, LS {target_ls}')

    if ref_res is not None:
        rdata, rx_min, rx_max, ry_min, ry_max, rentries, n_ls, rdataset = ref_res
        if ref_ls_min is None and ref_ls_max is None:
            ref_title = f'Reference: Run {ref_run}, full run'
        else:
            ref_title = f'Reference: Run {ref_run}, LS {ref_ls_min}-{ref_ls_max}'
        _plot_trackmap_data(rdata, rx_min, rx_max, ry_min, ry_max, ax=axs[1], run=ref_run, title=ref_title)

    if target_res is not None and ref_res is not None:
        data, x_min, x_max, y_min, y_max = target_res[0], target_res[1], target_res[2], target_res[3], target_res[4]
        rdata = ref_res[0]
        norm_target, target_scale = normalize_top_half(data)
        norm_ref, ref_scale = normalize_top_half(rdata)
        ratio = norm_target / norm_ref
        _plot_trackmap_data(ratio, x_min, x_max, y_min, y_max, ax=axs[2], run=target_run,
                             title='Ratio',
                             cmap='coolwarm', vmin=ratio_vmin, vmax=ratio_vmax,
                             cbar_label='Ratio (norm. target / norm. reference)')

        # Poisson uncertainty (sqrt of bin content) on each map, scaled by the
        # same normalization factor as the nominal values, combined in
        # quadrature to get the uncertainty on (norm_target - norm_ref).
        unc_target = np.sqrt(data) / target_scale
        unc_ref = np.sqrt(rdata) / ref_scale
        unc_diff = np.sqrt(unc_target**2 + unc_ref**2)
        significance = np.divide(
            norm_target - norm_ref, unc_diff,
            out=np.zeros_like(unc_diff), where=unc_diff > 0,
        )
        _plot_trackmap_data(significance, x_min, x_max, y_min, y_max, ax=axs[3], run=target_run,
                             title='Significance',
                             cmap='coolwarm', vmin=sig_vmin, vmax=sig_vmax,
                             cbar_label='(target - reference) / stat. unc.')

    fig.tight_layout()
    return fig, axs

In [ ]:
def save_trackmap(run, ls, plot_dir=PLOT_DIR, **kwargs):
    '''Plots a single (run, ls) track map and saves it to
    <plot_dir>/trackmap_<run>_<ls>.png. Returns the saved path, or None if
    no data was found.'''
    res = plot_trackmap(run, ls, **kwargs)
    if res is None:
        return None
    fig, ax, im = res
    os.makedirs(plot_dir, exist_ok=True)
    path = os.path.join(plot_dir, f'trackmap_{run}_{ls}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved to {os.path.abspath(path)}')
    return path

In [ ]:
def save_compare_trackmaps(target_run, target_ls, ref_run=None, ref_ls_min=None, ref_ls_max=None, plot_dir=PLOT_DIR, **kwargs):
    '''Plots and saves a target-vs-reference-vs-ratio comparison (see
    compare_trackmaps()) to
    <plot_dir>/trackmap_compare_<target_run>_<target_ls>_vs_<ref_tag>.png.
    Returns the saved path.'''
    fig, axs = compare_trackmaps(target_run, target_ls, ref_run=ref_run, ref_ls_min=ref_ls_min, ref_ls_max=ref_ls_max, **kwargs)
    os.makedirs(plot_dir, exist_ok=True)
    ref_run_tag = ref_run if ref_run is not None else target_run
    ref_tag = 'fullrun' if ref_ls_min is None and ref_ls_max is None else f'{ref_ls_min}-{ref_ls_max}'
    path = os.path.join(plot_dir, f'trackmap_compare_{target_run}_{target_ls}_vs_{ref_run_tag}_{ref_tag}.png')
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print(f'Saved to {os.path.abspath(path)}')
    return path

In [ ]:
# plot each configured (run, ls) in its own figure
for run, ls in RUNS_LS:
    plot_trackmap(run, ls)
    plt.show()

In [ ]:
# alternative: all configured (run, ls) as a grid of subplots
ncols = 3
nrows = -(-len(RUNS_LS) // ncols)  # ceil division
fig, axs = plt.subplots(nrows, ncols, figsize=(5*ncols, 5*nrows), squeeze=False)
for idx, (run, ls) in enumerate(RUNS_LS):
    ax = axs[idx // ncols][idx % ncols]
    plot_trackmap(run, ls, ax=ax)
for idx in range(len(RUNS_LS), nrows*ncols):
    axs[idx // ncols][idx % ncols].axis('off')
fig.tight_layout()
plt.show()

In [ ]:
# run all configured target-vs-reference-vs-ratio comparisons
for target_run, target_ls, ref_run, ref_ls_min, ref_ls_max in COMPARISONS:
    save_compare_trackmaps(target_run, target_ls, ref_run=ref_run, ref_ls_min=ref_ls_min, ref_ls_max=ref_ls_max)
    plt.show()